# 🎯 Module 13 · 05 — TF-IDF

> **Stage:** Machine Learning → NLP  
> **Level:** Beginner → Intermediate  
> **Module:** 13 — NLP Basics / 05 — TF-IDF  
> **Date:** 29 January 2026  
> **Previous:** 04 — Bag of Words  
> **Next:** 06 — Word Embeddings

---

**TF-IDF** stands for **Term Frequency — Inverse Document Frequency**.

> A word is important when it appears **frequently in a document** but **rarely across other documents**.

In this notebook we'll cover:

- Why TF-IDF exists
- **Term Frequency (TF)**
- **Inverse Document Frequency (IDF)**
- Combining TF and IDF
- `TfidfVectorizer` (scikit-learn)
- `fit_transform()` vs `transform()`
- N-grams, `min_df`, `max_df`, `max_features`
- Sublinear TF
- Normalization
- TF-IDF for classification, search, and similarity
- TF-IDF vs BoW
- Limitations

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain TF-IDF in your own words
- Compute **TF** and **IDF** by hand
- Combine them into TF-IDF scores
- Use `TfidfVectorizer` correctly
- Explain why we never `fit()` on test data
- Control the vocabulary with `min_df`, `max_df`, `max_features`
- Add **n-grams**
- Explain **sublinear TF** and **normalization**
- Use TF-IDF for **classification**, **search**, and **similarity**
- Distinguish **TF-IDF from BoW**
- Explain TF-IDF's **limitations**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Sample corpus used throughout the notebook
documents = [
    "I love Python",
    "I love machine learning",
    "Python is powerful",
    "Machine learning is powerful",
    "I love deep learning",
]

for i, doc in enumerate(documents, 1):
    print(f'Doc {i}: {doc}')

---
## 🧠 1. Why TF-IDF?

Bag of Words tells us:

```text
How many times does a word appear?
```

But **frequency alone** doesn't tell us how **informative** a word is.

Words such as:

```text
the
is
this
product
```

may appear **everywhere** — frequent but **not useful for distinguishing documents**.

**TF-IDF** reduces the importance of words that appear **across many documents**.

---
## 📈 2. Term Frequency (TF)

$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total terms in } d}$$

Example:

```text
Document:  "python python machine learning"
```

Total terms: 4

```text
TF(python)   = 2 / 4 = 0.5
TF(machine)  = 1 / 4 = 0.25
```

Terms that appear **more often within a document** get **more weight**.

In [ ]:
# Compute TF by hand for one document
doc = "python python machine learning"
tokens = doc.split()
total = len(tokens)

tf = {}
for t in set(tokens):
    tf[t] = tokens.count(t) / total

print('Tokens       :', tokens)
print('Total terms  :', total)
print('TF:')
for k, v in tf.items():
    print(f'  {k:<10} {v:.4f}')

---
## 📉 3. Inverse Document Frequency (IDF)

$$\text{IDF}(t) = \log\!\left(\frac{N}{df(t)}\right)$$

where

- `N` = total number of documents
- `df(t)` = number of documents containing term `t`

### Example

```text
Total documents = 100

Word appears in 5 documents:
  IDF = log(100/5) = log(20)   → high

Word appears in 90 documents:
  IDF = log(100/90) = log(1.11) → low
```

Rare words → **high IDF**  
Common words → **low IDF**

In [ ]:
# IDF by hand on our sample corpus
N = len(documents)

# df(t) = how many documents contain term t
df = {}
for doc in documents:
    for w in set(doc.lower().split()):
        df[w] = df.get(w, 0) + 1

# IDF = log(N / df)
idf = {w: math.log(N / d) for w, d in df.items()}

print(f'{"Term":<12} {"df":>4} {"IDF":>10}')
print('-' * 30)
for w in sorted(idf, key=lambda x: idf[x], reverse=True):
    print(f'{w:<12} {df[w]:>4} {idf[w]:>10.4f}')

print()
print('Words appearing in every doc (e.g. "love") get IDF ≈ 0.')

---
## ⚖️ 4. Combining TF and IDF

$$\text{TF-IDF} = \text{TF} \times \text{IDF}$$

```text
High TF + High IDF  →  High TF-IDF
```

A word gets a high score when:

```text
It is frequent in this document
         +
It is relatively rare across the corpus
```

A word that appears **in almost every document** gets a **lower** score.

In [ ]:
# Manually compute TF-IDF for each document
def compute_tf(doc):
    tokens = doc.lower().split()
    total = len(tokens)
    return {w: tokens.count(w) / total for w in set(tokens)}

def compute_idf(documents):
    N = len(documents)
    df = {}
    for doc in documents:
        for w in set(doc.lower().split()):
            df[w] = df.get(w, 0) + 1
    return {w: math.log(N / d) for w, d in df.items()}

idf = compute_idf(documents)

print('TF-IDF by document:')
for i, doc in enumerate(documents, 1):
    tf = compute_tf(doc)
    scores = {w: tf[w] * idf[w] for w in tf}
    print(f'\nDoc {i}: {doc}')
    for w in sorted(scores, key=lambda x: scores[x], reverse=True):
        print(f'   {w:<10}  TF={tf[w]:.3f}  IDF={idf[w]:.3f}  TF-IDF={scores[w]:.4f}')

---
## 💡 5. Simple Intuition

Imagine a collection of news articles.

The word:

```text
"the"
```

appears in almost every article — **low information**.

The word:

```text
"cryptocurrency"
```

appears frequently in a few technology articles but rarely elsewhere — **high information**.

```text
"the"             → low importance
"cryptocurrency"  → higher importance
```

This is the core intuition behind TF-IDF.

---
## 🧮 6. TF-IDF Matrix

TF-IDF creates a **numerical matrix**:

```text
             I    love   Python   machine  learning  ...
Doc 1       0.5  0.5    0.7      0        0
Doc 2       0.5  0.5    0        0.6      0.6
Doc 3       0    0      0.6      0        0         ...
```

Each value is the **TF-IDF weight** of a term for a specific document.

The matrix is usually **sparse** — most entries are zero.

In [ ]:
# Build the TF-IDF matrix with scikit-learn
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

df = pd.DataFrame(
    X.toarray().round(3),
    columns=vectorizer.get_feature_names_out(),
    index=[f'Doc{i+1}' for i in range(len(documents))],
)

print('TF-IDF matrix:')
print(df)

print('\nShape     :', X.shape)
print('Non-zeros :', X.nnz, '/', X.shape[0] * X.shape[1])
print(f'Sparsity  : {1 - X.nnz / (X.shape[0] * X.shape[1]):.1%}')

---
## 🛠️ 7. TF-IDF with scikit-learn

```python
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)
```

- Vocabulary:  `vectorizer.get_feature_names_out()`
- Matrix:      `X.toarray()`

Values are **TF-IDF weights** — not counts.

In [ ]:
# Vocabulary and a few sample weights
vocab = vectorizer.get_feature_names_out()
print('Vocabulary size:', len(vocab))
print('Vocabulary     :', list(vocab))

# Compare a couple of common vs rare terms
common = 'love' if 'love' in vocab else None
rare   = 'powerful' if 'powerful' in vocab else None

if common:
    print(f'\nDoc 1 weight for "{common}":', df.loc['Doc1', common])
if rare:
    print(f'Doc 3 weight for "{rare}":', df.loc['Doc3', rare])

---
## 🔀 8. `fit_transform()` vs `transform()`

### During training

```python
X_train = vectorizer.fit_transform(train_text)
```

This does **two** things:

```text
fit       → learn vocabulary and IDF statistics
transform → convert text into TF-IDF vectors
```

### For test data

```python
X_test = vectorizer.transform(test_text)
```

Do **not** call `fit()` on test data.

### ✅ Correct

```text
Training data  → fit_transform()
Testing  data  → transform()      ← only
```

> This prevents **data leakage**.

In [ ]:
# Demonstrate correct train / test usage
train = ["I love Python", "Machine learning is powerful"]
test  = ["I love deep learning"]

vec = TfidfVectorizer()
Xtr = vec.fit_transform(train)
Xte = vec.transform(test)

print('Train matrix shape:', Xtr.shape)
print('Test  matrix shape:', Xte.shape)
print()
print('Vocabulary learned only from TRAIN:')
print(list(vec.get_feature_names_out()))
print()
print('Test vector (all zero except known words):')
print(Xte.toarray().round(3))

---
## 🎛️ 9. TF-IDF Parameters

### `ngram_range`

```python
TfidfVectorizer(ngram_range=(1, 2))
```

Includes **unigrams + bigrams** — `"machine learning"` becomes a feature.

### `min_df`
Ignore very rare terms:
```python
TfidfVectorizer(min_df=2)
```

### `max_df`
Ignore extremely common terms:
```python
TfidfVectorizer(max_df=0.90)
```

### `max_features`
Limit vocabulary size:
```python
TfidfVectorizer(max_features=5000)
```

In [ ]:
# Compare vocabulary size under different settings
configs = [
    ('default',              TfidfVectorizer()),
    ('ngram_range=(1,2)',    TfidfVectorizer(ngram_range=(1, 2))),
    ('min_df=2',             TfidfVectorizer(min_df=2)),
    ('max_df=0.8',           TfidfVectorizer(max_df=0.8)),
    ('max_features=3',       TfidfVectorizer(max_features=3)),
    ('sublinear_tf=True',    TfidfVectorizer(sublinear_tf=True)),
    ('stop_words="english"', TfidfVectorizer(stop_words='english')),
]

print(f'{"Config":<26} {"Vocab":>6} {"Shape":>12}')
print('-' * 48)
for name, v in configs:
    Xc = v.fit_transform(documents)
    print(f'{name:<26} {len(v.get_feature_names_out()):>6} {str(Xc.shape):>12}')

---
## 📉 10. Sublinear TF

Normal TF treats words with more occurrences as **linearly more important**.

A common alternative uses:

```text
TF = 1 + log(tf)
```

In scikit-learn:

```python
TfidfVectorizer(sublinear_tf=True)
```

This compresses the effect of large counts — the jump from **1 → 10** occurrences is not treated as **10×** more important.

In [ ]:
# Compare linear TF vs sublinear TF on a heavy-repetition document
doc_repeat = ["word word word word word other"]

v_lin = TfidfVectorizer(sublinear_tf=False)
v_sub = TfidfVectorizer(sublinear_tf=True)

lin = v_lin.fit_transform(doc_repeat).toarray()[0]
sub = v_sub.fit_transform(doc_repeat).toarray()[0]

print(f'{"Term":<10} {"Linear":>10} {"Sublinear":>12}')
print('-' * 34)
for t, a, b in zip(v_lin.get_feature_names_out(), lin, sub):
    print(f'{t:<10} {a:>10.4f} {b:>12.4f}')

---
## 📐 11. Normalization

TF-IDF vectors are commonly **L2-normalized**.

```text
Raw TF-IDF Vector
       ↓
Normalization
       ↓
Normalized Vector
```

This makes vectors easier to compare — especially when using **cosine similarity**.

Scikit-learn's `TfidfVectorizer` uses **L2 normalization** by default.

In [ ]:
# Verify L2 normalization (each row has norm = 1)
norms = np.linalg.norm(X.toarray(), axis=1)
print('Row norms (should be ≈ 1):')
print(norms.round(6))

---
## 🏷️ 12. TF-IDF for Text Classification

Very common classical NLP pipeline:

```text
Text
 ↓
Preprocessing
 ↓
TF-IDF
 ↓
Logistic Regression
 ↓
Prediction
```

Example:

```text
Customer Review → TF-IDF → Logistic Regression → Positive / Negative
```

Works well with:

- Logistic Regression
- Linear SVM
- Naive Bayes

In [ ]:
# End-to-end classification with TF-IDF + Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

positive = [
    "I love this product", "amazing quality", "absolutely wonderful",
    "best purchase ever", "highly recommend", "fantastic experience",
    "great service", "excellent quality",
]
negative = [
    "worst product ever", "terrible quality", "absolutely awful",
    "waste of money", "do not recommend", "horrible experience",
    "bad service", "poor quality",
]

texts = positive + negative
labels = [1] * len(positive) + [0] * len(negative)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000)),
])

pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, preds), 3))
print()
print(classification_report(y_test, preds, target_names=['neg', 'pos']))

# Predict new text
new = ["absolutely fantastic product"]
print('New prediction:', pipe.predict(new)[0], '(1 = positive, 0 = negative)')

---
## 🔍 13. TF-IDF for Search

TF-IDF can power a **simple classical search engine**.

```text
Query
   ↓
TF-IDF Vector
   ↓
Compare with document vectors
   ↓
Rank by cosine similarity
   ↓
Most relevant documents
```

In [ ]:
# Tiny search engine with TF-IDF + cosine similarity
docs = [
    "Python machine learning tutorial",
    "Java web development guide",
    "Machine learning with Python",
    "Deep learning with neural networks",
    "Football match highlights today",
]

vec = TfidfVectorizer()
X_docs = vec.fit_transform(docs)

def search(query, top_k=3):
    X_q = vec.transform([query])
    sims = cosine_similarity(X_q, X_docs).ravel()
    ranked = sorted(enumerate(sims), key=lambda x: -x[1])
    return [(docs[i], round(score, 4)) for i, score in ranked[:top_k]]

query = "python machine learning"
print(f'Query: {query!r}')
print('Top results:')
for doc, score in search(query):
    print(f'  {score:>6.4f}  {doc}')

---
## 📐 14. TF-IDF for Similarity

Suppose:

```text
Text A:  "machine learning tutorial"
Text B:  "machine learning course"
```

Their TF-IDF vectors share important features:

```text
machine
learning
```

**Cosine similarity** of their TF-IDF vectors gives a **similarity score**.

Higher = more similar.

In [ ]:
# Cosine similarity between text pairs using TF-IDF
pairs = [
    ("machine learning tutorial", "machine learning course"),
    ("machine learning tutorial", "football match today"),
    ("python programming", "python tutorial"),
    ("cat sat on mat", "dog played in yard"),
]

print(f'{"Text A":<30} {"Text B":<30} {"Sim":>6}')
print('-' * 72)
for a, b in pairs:
    vec = TfidfVectorizer()
    Xab = vec.fit_transform([a, b])
    sim = cosine_similarity(Xab[0], Xab[1])[0, 0]
    print(f'{a:<30} {b:<30} {sim:>6.3f}')

---
## ⚖️ 15. TF-IDF vs Bag of Words

| Feature | Bag of Words | TF-IDF |
| --- | --- | --- |
| Numerical representation | ✅ | ✅ |
| Word counts | ✅ | Uses TF |
| Considers corpus frequency | ❌ | ✅ |
| Downweights common words | ❌ | ✅ |
| Sparse | ✅ | ✅ |
| Easy to implement | ✅ | ✅ |
| Useful for classification | ✅ | ✅ |
| Useful for search | ✅ | ✅ |

### Simple mental model

```text
BoW     → "How often does this word appear?"
TF-IDF  → "How important is this word for this document?"
```

In [ ]:
# Compare BoW vs TF-IDF for the term 'love' in Doc 1
bow = CountVectorizer().fit_transform(documents)
tfidf = TfidfVectorizer().fit_transform(documents)

term = 'love'
bow_vec = CountVectorizer().fit(documents)
tfidf_vec = TfidfVectorizer().fit(documents)

bi = list(bow_vec.get_feature_names_out()).index(term)
ti = list(tfidf_vec.get_feature_names_out()).index(term)

print(f'Term: "{term}"')
print(f'BoW   weights :', bow.toarray()[:, bi].tolist())
print(f'TF-IDF weights:', tfidf.toarray()[:, ti].round(3).tolist())
print()
print('BoW counts appearances. TF-IDF adjusts for rarity across documents.')

---
## ⚠️ 16. Limitations of TF-IDF

### No deep semantic understanding

```text
car
automobile
vehicle
```

treated as **separate** features — TF-IDF doesn't understand they're related.

### Limited context

```text
"I like this product"
"I do not like this product"
```

The model can use `not` as a feature, but TF-IDF itself has **no semantic model** of negation.

### Large vocabulary

Large datasets → high-dimensional sparse matrices.

### No true meaning

A TF-IDF vector represents **word importance**, not **semantic meaning**.

> These limitations motivated **word embeddings** and modern neural NLP.

---
## 🧪 17. Practical End-to-End Example

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

texts = [
    "I love this product",
    "This product is amazing",
    "I hate this product",
    "This product is terrible",
]
labels = ["positive", "positive", "negative", "negative"]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

model = LogisticRegression()
model.fit(X, labels)

new_text = ["I love this amazing product"]
X_new = vectorizer.transform(new_text)
prediction = model.predict(X_new)
print(prediction)
```

### Architecture

```text
Text
 ↓
TF-IDF Vectorization
 ↓
Numerical Features
 ↓
Logistic Regression
 ↓
Prediction
```

> In a real project: split the data first, put vectorizer + model in a **Pipeline**.

---
## 🧠 18. Mental Model

```text
                DOCUMENT
                   ↓
          ┌─────────────────┐
          │ Term Frequency  │
          └────────┬────────┘
                   │
                   ×
          ┌────────▼────────┐
          │  Inverse        │
          │  Document       │
          │  Frequency      │
          └────────┬────────┘
                   ↓
               TF-IDF
                   ↓
          Numerical Vector
                   ↓
              ML / Search
```

### 🧭 Golden Rule

> **TF-IDF gives higher importance to terms that are useful in a document but relatively uncommon across the document collection.**

---
## 🏋️ 19. Hands-On Exercises

### Exercise 1 — TF by hand
Document: `"cat cat dog bird bird bird"`

Compute TF for `cat`, `dog`, and `bird`.

### Exercise 2 — IDF by hand
Given 10 documents:
- Word `A` appears in 2 documents
- Word `B` appears in 8 documents

Compute IDF for each. Which is higher, and why?

### Exercise 3 — TF-IDF intuition
Two words:
- Word X: TF = 0.5, IDF = 0.1
- Word Y: TF = 0.3, IDF = 2.5

Which has a higher TF-IDF score? What does this mean for the document?

### Exercise 4 — Vocabulary control
Build a TF-IDF matrix for a dataset with `min_df=2`, `max_df=0.8`, `max_features=100`.  
Compare vocabulary size to the default.

### Exercise 5 — Search engine
Build a small search engine over these docs:
```python
docs = [
    "python programming tutorial",
    "java development guide",
    "machine learning with python",
    "deep learning neural networks",
]
query = "python learning"
```
Rank the documents by cosine similarity.

### Exercise 6 — Sublinear TF
Compare `sublinear_tf=True` vs `False` on a document with heavy repetition.  
Which gives more balanced weights?

---
## 🎤 20. Interview Questions

1. **What does TF-IDF stand for?**  
   Term Frequency — Inverse Document Frequency.

2. **What does TF measure?**  
   How often a term appears in a document, normalized by the document length.

3. **What does IDF measure?**  
   How rare a term is across the entire corpus.

4. **Why does IDF reduce common words?**  
   Common words don't help distinguish documents, so their weight should be low.

5. **What is the TF-IDF formula?**  
   `TF × IDF`.

6. **Why does TF-IDF work well with linear models?**  
   It produces high-dimensional, sparse features that linear models handle efficiently.

7. **Why not fit the vectorizer on the full dataset?**  
   It causes data leakage — test data influences vocabulary and IDF.

8. **What does `min_df` do?**  
   Ignores terms appearing in fewer than `min_df` documents.

9. **What does `max_df` do?**  
   Ignores terms appearing in more than `max_df` fraction of documents.

10. **What is sublinear TF?**  
    A logarithmic scaling of TF: `1 + log(tf)`.

11. **Why normalize TF-IDF vectors?**  
    So cosine similarity between vectors is comparable across documents of different lengths.

12. **TF-IDF for search — how?**  
    Represent query and documents as TF-IDF vectors and rank by cosine similarity.

13. **Main limitation of TF-IDF?**  
    No deep semantic understanding — `car` and `automobile` are unrelated features.

14. **BoW vs TF-IDF?**  
    BoW counts; TF-IDF weights by document-vs-corpus importance.

15. **How do you avoid leakage with TF-IDF?**  
    Split first, fit on training only, or wrap it in a scikit-learn `Pipeline`.

---
## 🏁 21. Key Takeaways

### You now understand

- What TF-IDF means
- Term Frequency
- Inverse Document Frequency
- The TF-IDF formula
- Why common words receive lower weight
- `TfidfVectorizer`
- `fit_transform()` vs `transform()`
- N-grams with TF-IDF
- `min_df`, `max_df`, `max_features`
- Sublinear TF
- Normalization
- TF-IDF for classification
- TF-IDF for search
- TF-IDF for similarity
- TF-IDF vs BoW
- TF-IDF limitations
- Why TF-IDF works well with classical ML

### The progression

```text
Text
 ↓
Vocabulary
 ↓
TF × IDF
 ↓
TF-IDF Matrix
 ↓
ML / Search / Similarity
```

---

### 🔗 What's Next?

**06 — Word Embeddings**

We move from **word-frequency-based representations → dense numerical representations that can capture relationships between words**.